# HDF5 vector/scalar inspector

Inspect raw datasets in an analysis HDF5 file to debug NaNs after DataFrame extraction.

Run the cell below after setting `H5_PATH` and `COLUMNS` as needed.


In [1]:
import hdf5plugin
from pathlib import Path
import h5py
import numpy as np
from ddstartup.utils.io_functions import resolve_h5_inputs
ROOT = Path.cwd().parent.parent

H5_SPEC = "outputs/paper_parametric_T_seeded"  # folder, file path, or "latest"
COLUMNS = ["I_target", "P_DT_eq", "P_aux", "t_startup", "unrealized_profits"]
FILTER_SUCCESS = True
MAX_PRINT = 25

# ------------------------

# Resolve HDF5 file (supports folder/latest spec)
resolved, _ = resolve_h5_inputs(H5_SPEC, root=ROOT)
if not resolved:
    raise FileNotFoundError(f"No .h5 found for spec: {H5_SPEC}")
h5_path = resolved[0]
print(f"Inspecting: {h5_path}")

with h5py.File(h5_path, "r") as h5:
    all_keys = list(h5.keys())
    print(f"Datasets found ({len(all_keys)}): {all_keys}")
    targets = list(COLUMNS) if COLUMNS else all_keys
    mask = None
    if FILTER_SUCCESS and "sol_success" in h5:
        mask = np.asarray(h5["sol_success"][...]).astype(bool)
        print(f"sol_success present: {mask.sum()} / {len(mask)} successful")

    for col in targets:
        if col not in h5:
            print(f"\nColumn '{col}' not found.")
            continue
        data = h5[col][...]
        print(f"\nColumn: {col}")
        print(f"  shape: {data.shape}")
        print(f"  dtype: {data.dtype}")
        if data.ndim == 1 and mask is not None and len(mask) == len(data):
            data = data[mask]
            print(f"  filtered to sol_success=True -> {data.shape[0]} rows")
        else:
            print("  not filtered (mask length mismatch or vector data)")
        flat = np.ravel(data)
        shown = flat[:MAX_PRINT]
        print(f"  sample values (first {len(shown)}): {shown}")
        if flat.size > len(shown):
            print(f"  ... ({flat.size - len(shown)} more)")


Inspecting: /home/alessmor/Scrivania/dd_startup/outputs/paper_parametric_T_seeded/ddstartup_20251113_002935_parametric_T_seeded.h5
Datasets found (35): ['E_lost', 'I_target', 'N_ifc', 'N_ofc', 'N_stor', 'P_DDn', 'P_DDp', 'P_DT', 'P_DT_eq', 'P_aux', 'P_aux_DT_eq', 'Q_DD', 'Q_DT_eq', 'TBE', 'TBR_DDn', 'TBR_DT', 'T_i', 'V_plasma', 'capacity_factor', 'error', 'eta_th', 'linear_index', 'n_D', 'n_He3', 'n_T', 'n_tot', 'parameter_fields', 'price_of_electricity', 'sol_success', 't_startup', 'tau_ifc', 'tau_ofc', 'tau_p_He3', 'tau_p_T', 'unrealized_profits']
sol_success present: 5943770 / 6237000 successful

Column: I_target
  shape: (6237000,)
  dtype: float64
  filtered to sol_success=True -> 5943770 rows
  sample values (first 25): [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan]
  ... (5943745 more)

Column: P_DT_eq
  shape: (6237000,)
  dtype: float64
  filtered to sol_success=True -> 5943770 rows
  sample values (first 25): [39242642.3